### 4.6 National distribution of mobility infrastructure regimes

We next mapped the recovered regimes across Brazil. The purpose of this map was not to rank places by total traffic. Instead, it showed how localized mobility-covariance patterns partitioned the national territory into structurally distinct mobility environments.

This distinction mattered for interpretation. Two places could display comparable movement intensity while belonging to different regimes because visitation volume, repeat use, dwell time, and temporal stability are combined differently across space.

In [ ]:
# ============================================================
# PUBLICATION CARTOGRAPHY HELPERS
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, TwoSlopeNorm, ListedColormap
import numpy as np

plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.titleweight"] = "bold"


def add_north_arrow(ax, x, y, length, color="#2b2b2b"):
    ax.annotate(
        "",
        xy=(x, y + length),
        xytext=(x, y),
        arrowprops=dict(
            facecolor=color,
            edgecolor=color,
            width=2.0,
            headwidth=9,
            headlength=11,
        ),
        zorder=20,
    )
    ax.text(
        x,
        y + length + 0.015 * length,
        "N",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
        color=color,
        zorder=21,
    )


def add_scale_bar(ax, x, y, segment_length_m, n_segments=2, bar_height=10000, text_color="#2b2b2b"):
    for i in range(n_segments):
        face = "#2b2b2b" if i % 2 == 0 else "white"
        ax.add_patch(
            mpatches.Rectangle(
                (x + i * segment_length_m, y),
                segment_length_m,
                bar_height,
                facecolor=face,
                edgecolor="#2b2b2b",
                linewidth=0.6,
                zorder=20,
            )
        )

    ax.text(x, y + 2.2 * bar_height, "0", ha="center", va="bottom", fontsize=8.5, color=text_color)
    ax.text(
        x + segment_length_m,
        y + 2.2 * bar_height,
        f"{int(segment_length_m/1000)}",
        ha="center",
        va="bottom",
        fontsize=8.5,
        color=text_color,
    )
    ax.text(
        x + n_segments * segment_length_m,
        y + 2.2 * bar_height,
        f"{int(n_segments * segment_length_m/1000)} km",
        ha="center",
        va="bottom",
        fontsize=8.5,
        color=text_color,
    )


def finalize_publication_map(
    fig,
    ax,
    title,
    subtitle,
    footnote,
    left=0.08,
    title_y=0.955,
    subtitle_y=0.928,
    footnote_y=0.035,
):
    ax.set_axis_off()

    fig.text(
        left,
        title_y,
        title,
        fontsize=18,
        fontweight="bold",
        ha="left",
        va="top",
    )
    fig.text(
        left,
        subtitle_y,
        subtitle,
        fontsize=11,
        color="#4f4f4f",
        ha="left",
        va="top",
    )
    fig.text(
        left,
        footnote_y,
        footnote,
        fontsize=8.8,
        color="#5a5a5a",
        ha="left",
        va="bottom",
    )

In [ ]:
# ============================================================
# NATIONAL MOBILITY INFRASTRUCTURE REGIMES
# Publication-cartography version with South America context
# ============================================================

import matplotlib.patches as mpatches
import geopandas as gpd

plot_crs = "EPSG:5880"

gdf_reg = gdf.dropna(subset=["infrastructure_regime"]).copy()
gdf_reg["infrastructure_regime"] = gdf_reg["infrastructure_regime"].astype(int)
gdf_reg = gdf_reg.to_crs(plot_crs)

# Fix invalid geometries before dissolve
gdf_reg["geometry"] = gdf_reg.geometry.buffer(0)
brazil_outline = gdf_reg[["geometry"]].dissolve()

# ------------------------------------------------------------
# South America context layer
# ------------------------------------------------------------
try:
    world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
    south_america = world.loc[world["continent"] == "South America"].to_crs(plot_crs).copy()
except Exception:
    south_america = None
    print("South America context layer could not be loaded. Proceeding without it.")

# ------------------------------------------------------------
# Publication-safe categorical palette
# ------------------------------------------------------------
regime_palette = {
    1: "#355070",  # deep muted blue
    2: "#6D597A",  # muted violet
    3: "#B56576",  # dusty rose
    4: "#E9C46A",  # warm sand
}

regime_labels = {
    1: "Regime 1",
    2: "Regime 2",
    3: "Regime 3",
    4: "Regime 4",
}

# ------------------------------------------------------------
# Figure
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(13.2, 12.8), facecolor="white")
ax.set_facecolor("#fbfbfb")

# Context layer first
if south_america is not None:
    south_america.plot(
        ax=ax,
        color="#efefef",
        edgecolor="#d0d0d0",
        linewidth=0.35,
        zorder=1,
    )

# Main thematic layer
gdf_reg.plot(
    color=gdf_reg["infrastructure_regime"].map(regime_palette),
    linewidth=0.008,
    edgecolor="#cfcfcf",
    ax=ax,
    zorder=3,
)

# Brazil outline on top
brazil_outline.boundary.plot(
    ax=ax,
    color="#444444",
    linewidth=0.65,
    zorder=5,
)

# ------------------------------------------------------------
# Map extent with margin around Brazil
# ------------------------------------------------------------
xmin, ymin, xmax, ymax = gdf_reg.total_bounds
xrange = xmax - xmin
yrange = ymax - ymin

ax.set_xlim(xmin - 0.10 * xrange, xmax + 0.06 * xrange)
ax.set_ylim(ymin - 0.08 * yrange, ymax + 0.06 * yrange)

ax.set_axis_off()

# ------------------------------------------------------------
# Title and subtitle
# ------------------------------------------------------------
fig.text(
    0.08,
    0.955,
    "National Mobility Infrastructure Regimes",
    fontsize=18,
    fontweight="bold",
    ha="left",
    va="top",
)

fig.text(
    0.08,
    0.928,
    "Localized covariance signatures identified four spatially differentiated mobility environments across Brazil",
    fontsize=11,
    color="#4f4f4f",
    ha="left",
    va="top",
)

# ------------------------------------------------------------
# Legend
# ------------------------------------------------------------
legend_handles = [
    mpatches.Patch(facecolor=regime_palette[i], edgecolor="none", label=regime_labels[i])
    for i in sorted(regime_palette)
]

legend = ax.legend(
    handles=legend_handles,
    title="Infrastructure Regime",
    loc="lower left",
    bbox_to_anchor=(0.02, 0.12),
    frameon=True,
    framealpha=1.0,
    facecolor="white",
    edgecolor="#d0d0d0",
    fontsize=10,
    title_fontsize=11,
    borderpad=0.8,
    labelspacing=0.55,
    handlelength=1.5,
    handletextpad=0.6,
)
legend._legend_box.align = "left"

# ------------------------------------------------------------
# North arrow
# ------------------------------------------------------------
arrow_x = xmin + 0.94 * xrange
arrow_y = ymin + 0.13 * yrange
arrow_len = 0.07 * yrange

ax.annotate(
    "",
    xy=(arrow_x, arrow_y + arrow_len),
    xytext=(arrow_x, arrow_y),
    arrowprops=dict(
        facecolor="#2b2b2b",
        edgecolor="#2b2b2b",
        width=2.0,
        headwidth=9,
        headlength=11,
    ),
    zorder=10,
)

ax.text(
    arrow_x,
    arrow_y + arrow_len + 0.018 * yrange,
    "N",
    ha="center",
    va="bottom",
    fontsize=10,
    fontweight="bold",
    color="#2b2b2b",
)

# ------------------------------------------------------------
# Scale bar
# ------------------------------------------------------------
scale_bar_km = 1000
scale_bar_m = scale_bar_km * 1000

bar_x = xmin + 0.06 * xrange
bar_y = ymin + 0.05 * yrange
bar_h = 0.004 * yrange

ax.add_patch(
    mpatches.Rectangle(
        (bar_x, bar_y),
        scale_bar_m,
        bar_h,
        facecolor="#2b2b2b",
        edgecolor="#2b2b2b",
        linewidth=0.6,
        zorder=10,
    )
)

ax.add_patch(
    mpatches.Rectangle(
        (bar_x + scale_bar_m, bar_y),
        scale_bar_m,
        bar_h,
        facecolor="white",
        edgecolor="#2b2b2b",
        linewidth=0.6,
        zorder=10,
    )
)

ax.text(bar_x, bar_y + 2.0 * bar_h, "0", ha="center", va="bottom", fontsize=8.5, color="#2b2b2b")
ax.text(
    bar_x + scale_bar_m,
    bar_y + 2.0 * bar_h,
    f"{scale_bar_km}",
    ha="center",
    va="bottom",
    fontsize=8.5,
    color="#2b2b2b",
)
ax.text(
    bar_x + 2 * scale_bar_m,
    bar_y + 2.0 * bar_h,
    f"{2 * scale_bar_km} km",
    ha="center",
    va="bottom",
    fontsize=8.5,
    color="#2b2b2b",
)

# ------------------------------------------------------------
# Footnote
# ------------------------------------------------------------
fig.text(
    0.08,
    0.038,
    "Source: analysis-ready tract-level mobility asset. Regimes were assigned by propagating localized PCA signatures from systematically distributed focal locations.",
    ha="left",
    va="bottom",
    fontsize=8.8,
    color="#5a5a5a",
)

plt.tight_layout(rect=[0.02, 0.06, 0.98, 0.90])
plt.show()